# Empirical-null screening

This notebook runs the manuscript empirical-null screening stage against the resolved case-study artifacts. During automated validation it uses the deterministic demo dataset; once local real-data paths are configured it screens the real feature catalog against the retained PCA component scores.

## Notebook Overview

### Purpose
Screen features via empirical null hypothesis testing (Delta-null mechanism). Identify features with significant associations after accounting for null distribution.

### Inputs
- `dfm_conditioned`: Conditioned feature DataFrame from stage 02
- `df_pheno`: Phenotype/outcome variable
- Null distribution reference: `src/rfm_pipeline/null_distribution.py` (Delta-null workflow)
- Significance level: from configuration (default α=0.05)

### Outputs
- `null_distribution_results`: Summary of null hypothesis results (p-values, test statistics)
- Null distribution CSV: `outputs/03_empirical_null/null_distribution.csv`
- Significant features list: `outputs/03_empirical_null/features_03_after_null_screen.csv`
- Stage 3 handoff CSV: features passing null hypothesis test

### Parameters & Configuration
- **Null sample size**: Number of permutations for null distribution. Default 1000. Higher = more precise but slower.
- **Significance level (α)**: Rejection threshold. Default 0.05. Stricter thresholds (e.g., 0.01) are more conservative.
- **Delta-null correction**: Boolean. If True, applies bias-corrected null distribution. Default True.
- **Test type**: Univariate (default) or adjusted for multiple comparisons (Bonferroni, BH-FDR).

### Expected Output
Notebook completes in ~30-60 seconds (depends on null sample size and feature count).
- Null distribution CSV written with permutation results
- Significant features: typically 50-150 features pass at α=0.05 (depends on signal strength)
- Stage 3 handoff CSV written
- No NaN in p-values or test statistics

---


In [ ]:
import os
import sys
from pathlib import Path


def _find_study_root() -> Path:
    env_root = os.environ.get("RFM_STUDY_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "configs").exists():
            return candidate
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "configs").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the study root. Set RFM_STUDY_ROOT or run from the study directory."
    )


REPO_ROOT = _find_study_root()
SRC_DIR = REPO_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from rfm_pipeline import (
    build_manuscript_notebook_context,
    run_empirical_null_screening_stage,
)

context = build_manuscript_notebook_context(REPO_ROOT, "03_empirical_null_screen.ipynb")
stage_result = run_empirical_null_screening_stage(context)
summary = stage_result.screening.summary
summary

### Step 1: Generate Null Distribution

Run empirical null screening using Delta-null mechanism.

**Process:**
1. For each feature, compute association with outcome (e.g., correlation, beta)
2. Generate null distribution via permutation (~1000 runs)
3. Compute empirical p-value: fraction of null permutations ≥ observed statistic
4. Repeat for all features in parallel (if available)

**Parameters:**
- `null_sample_size`: Number of permutations (default 1000)
- `alpha`: Significance level (default 0.05)
- `delta_null_correction`: Apply bias correction (default True)

**Output:**
- `null_results`: DataFrame with p-values, test statistics, significance flags

### Step 2: Filter to Significant Features

Keep only features with p-value < α. Export handoff CSV.

**Selection:**
- Features with empirical p-value < α are significant
- Typically ~5-15% of input features pass at α=0.05
- Export selected features and null distribution summary